# 07b - Relación entre las variables de calidad y el porcentaje de llenado

Examina si las variables de calidad del agua presentan asociación con el porcentaje de llenado de los embalses, con el fin de anticipar qué cabe esperar del escenario que las incorpora antes de abordar el modelado.

## Planteamiento

Una correlación calculada sobre el conjunto de embalses y sin descontar la estacionalidad puede reflejar asociaciones que no responden a ninguna relación temporal entre las variables. El análisis se plantea por ello en tres niveles de exigencia creciente:

1. **Correlación agregada**, sobre el conjunto de observaciones de todos los embalses.
2. **Correlación por embalse y desestacionalizada**, eliminando de ambas series el ciclo anual mediante regresión sobre los términos armónicos. Aísla la relación temporal dentro de cada serie de la variación entre embalses y de la estacionalidad compartida.
3. **Correlación con la variación futura del llenado** a 7, 30 y 90 días. Es la magnitud sobre la que las variables de calidad deberían aportar información si la hipótesis del trabajo se sostiene, dado que el nivel actual del embalse ya está disponible como predictor en todos los escenarios.

## 1. Configuración

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")

DIR_PROCESSED = Path("../data/processed")
DIR_FIGURAS = Path("../outputs/figures")

HORIZONTES = [7, 30, 90]
MIN_OBS = 365          # observaciones mínimas por embalse para calcular correlación
PARAMETROS = ["amonio_mgl", "conductividad_uscm", "oxigeno_mgl",
              "ph", "temp_agua_c", "turbidez_ntu"]

dataset = pd.read_parquet(DIR_PROCESSED / "dataset_modelado.parquet")
experimentales = set(pd.read_parquet(
    DIR_PROCESSED / "embalses_experimento_calidad.parquet")["ID_SAIH"])

df = dataset[dataset["ID_SAIH"].isin(experimentales)].copy()

print(f"Observaciones: {len(df):,} | embalses: {df['ID_SAIH'].nunique()}")

Observaciones: 111,758 | embalses: 17


## 2. Construcción del bloque único de calidad

Las estaciones de calidad asignadas a cada embalse se sitúan aguas arriba o aguas abajo según los casos, y solo cuatro embalses disponen de ambas. Mantener bloques separados genera columnas estructuralmente vacías en la mayoría de los embalses, lo que impide un tratamiento homogéneo.

Se construye por ello un bloque único de variables de calidad, tomando el registro de la estación situada aguas arriba cuando existe y el de la estación aguas abajo en su defecto. La prioridad se otorga a la posición aguas arriba por su mayor sentido físico como predictora: mide agua que aún no ha alcanzado el embalse, mientras que la situada aguas abajo mide agua ya regulada.

La procedencia se conserva en una variable categórica, de modo que los modelos puedan distinguir el tipo de estación y el análisis de resultados pueda desglosarse según ella.

In [2]:
for p in PARAMETROS:
    df[f"cal_{p}"] = df[f"cal_arriba_{p}"].fillna(df[f"cal_abajo_{p}"])

df["tipo_estacion"] = np.select(
    [df["cal_arriba_ph"].notna(), df["cal_abajo_ph"].notna()],
    ["arriba", "abajo"],
    default=None,
)

CAL_COLS = [f"cal_{p}" for p in PARAMETROS]

print("Cobertura del bloque único:")
for c in CAL_COLS:
    print(f"  {c:<28} {df[c].notna().mean()*100:5.1f}%")

print("\nProcedencia del registro:")
print(df["tipo_estacion"].value_counts(dropna=False).to_string())

print("\nEmbalses por procedencia predominante:")
print(df.groupby("ID_SAIH")["tipo_estacion"]
      .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)
      .value_counts().to_string())

Cobertura del bloque único:
  cal_amonio_mgl                86.8%
  cal_conductividad_uscm        95.3%
  cal_oxigeno_mgl               95.0%
  cal_ph                        95.4%
  cal_temp_agua_c               95.5%
  cal_turbidez_ntu              92.3%

Procedencia del registro:
tipo_estacion
abajo     62982
arriba    43606
NaN        5170

Embalses por procedencia predominante:
tipo_estacion
abajo     10
arriba     7


## 3. Correlación agregada

Primera aproximación, sobre el conjunto de observaciones de los diecisiete embalses y sin descontar el ciclo anual.

In [3]:
corr_agregada = (df[CAL_COLS + ["pct_llenado"]].corr()["pct_llenado"]
                 .drop("pct_llenado").sort_values(key=abs, ascending=False))

print("Correlación con el porcentaje de llenado:")
print(corr_agregada.round(3).to_string())

Correlación con el porcentaje de llenado:
cal_oxigeno_mgl           0.187
cal_ph                    0.092
cal_conductividad_uscm   -0.082
cal_amonio_mgl           -0.056
cal_temp_agua_c          -0.052
cal_turbidez_ntu          0.042


## 4. Correlación por embalse y desestacionalizada

Se calcula la correlación dentro de cada embalse por separado, sobre las series residuales tras eliminar de ambas el ciclo anual. El estadístico reportado es la mediana de los coeficientes obtenidos en los distintos embalses.

In [4]:
def residuo_estacional(d, col):
    """Serie residual tras eliminar el ciclo anual mediante regresión armónica."""
    X = np.column_stack([np.ones(len(d)), d["dia_anio_sin"], d["dia_anio_cos"]])
    beta = np.linalg.lstsq(X, d[col], rcond=None)[0]
    return d[col] - X @ beta

def corr_por_embalse(df, col_y, col_x, min_obs=MIN_OBS, desestacionalizar=True):
    """Coeficientes de correlación calculados individualmente por embalse."""
    rs = {}
    for emb, g in df.groupby("ID_SAIH"):
        d = g[[col_y, col_x, "dia_anio_sin", "dia_anio_cos"]].dropna()
        if len(d) < min_obs:
            continue
        if desestacionalizar:
            ry, rx = residuo_estacional(d, col_y), residuo_estacional(d, col_x)
        else:
            ry, rx = d[col_y], d[col_x]
        rs[emb] = np.corrcoef(ry, rx)[0, 1]
    return pd.Series(rs)

filas = []
for c in CAL_COLS:
    bruta = corr_por_embalse(df, "pct_llenado", c, desestacionalizar=False)
    desest = corr_por_embalse(df, "pct_llenado", c)
    filas.append({
        "variable": c.replace("cal_", ""),
        "n_embalses": len(desest),
        "agregada": round(corr_agregada[c], 3),
        "por_embalse": round(bruta.median(), 3),
        "desestacionalizada": round(desest.median(), 3),
    })

niveles = pd.DataFrame(filas).sort_values("agregada", key=abs, ascending=False)
print(niveles.to_string(index=False))

          variable  n_embalses  agregada  por_embalse  desestacionalizada
       oxigeno_mgl          17     0.187        0.113               0.048
                ph          17     0.092        0.044              -0.028
conductividad_uscm          17    -0.082       -0.170              -0.020
        amonio_mgl          17    -0.056        0.016               0.018
       temp_agua_c          17    -0.052       -0.066              -0.008
      turbidez_ntu          17     0.042        0.092               0.075


## 5. Correlación con la variación futura del llenado

El nivel actual del embalse está disponible como predictor en todos los escenarios, por lo que la contribución de las variables de calidad debería manifestarse sobre la variación futura del llenado y no sobre su nivel. Se calcula por ello la correlación con el incremento del porcentaje de llenado a cada uno de los horizontes de predicción, igualmente por embalse y sobre series desestacionalizadas.

In [5]:
for h in HORIZONTES:
    df[f"delta_h{h}"] = df.groupby("ID_SAIH")["pct_llenado"].shift(-h) - df["pct_llenado"]

filas = []
for c in CAL_COLS:
    fila = {"variable": c.replace("cal_", "")}
    for h in HORIZONTES:
        rs = corr_por_embalse(df, f"delta_h{h}", c)
        fila[f"h{h}"] = round(rs.median(), 3)
        fila[f"h{h}_max"] = round(rs.abs().max(), 3)
        fila["n_embalses"] = len(rs)
    filas.append(fila)

variacion = pd.DataFrame(filas).sort_values("h30", key=abs, ascending=False)
print(variacion[["variable", "n_embalses"] +
                [f"h{h}" for h in HORIZONTES] +
                [f"h{h}_max" for h in HORIZONTES]].to_string(index=False))

          variable  n_embalses     h7    h30    h90  h7_max  h30_max  h90_max
      turbidez_ntu          17 -0.001 -0.043 -0.052   0.155    0.166    0.200
conductividad_uscm          17  0.006  0.038  0.021   0.303    0.150    0.140
       oxigeno_mgl          17 -0.005 -0.013 -0.053   0.112    0.222    0.213
       temp_agua_c          17 -0.001 -0.010  0.030   0.128    0.101    0.205
        amonio_mgl          17  0.002 -0.003  0.005   0.058    0.074    0.095
                ph          17  0.008  0.001 -0.000   0.231    0.089    0.101


## 6. Dispersión entre embalses

La mediana puede ocultar comportamientos heterogéneos. Se examina la distribución de los coeficientes entre embalses para descartar que existan casos individuales con asociación apreciable.

In [6]:
H_DETALLE = 30

detalle = pd.DataFrame({
    c.replace("cal_", ""): corr_por_embalse(df, f"delta_h{H_DETALLE}", c)
    for c in CAL_COLS
}).round(3)

print(f"Correlación con la variación a {H_DETALLE} días, por embalse:")
print(detalle.to_string())
print(f"\nCoeficiente de mayor magnitud observado: {detalle.abs().max().max():.3f}")

Correlación con la variación a 30 días, por embalse:
      amonio_mgl  conductividad_uscm  oxigeno_mgl     ph  temp_agua_c  turbidez_ntu
E002       0.001               0.060        0.025  0.089       -0.060         0.010
E008      -0.012              -0.045        0.010 -0.039       -0.020         0.019
E009      -0.060              -0.057       -0.081 -0.081        0.017        -0.047
E011       0.018               0.039       -0.001  0.042        0.021        -0.030
E025      -0.027               0.128       -0.219  0.016        0.056        -0.165
E026      -0.039               0.150       -0.222  0.031        0.093        -0.166
E027      -0.003               0.043       -0.053  0.027        0.022        -0.043
E028       0.035              -0.148       -0.122 -0.085        0.101         0.047
E029       0.060               0.038        0.011  0.011        0.039        -0.015
E030      -0.001              -0.002       -0.049 -0.008       -0.079        -0.054
E031       0.002       

## 7. Conclusiones

La correlación agregada entre el oxígeno disuelto y el porcentaje de llenado alcanza 0,187, pero se reduce a 0,113 al calcularla dentro de cada embalse y a 0,048 tras eliminar el ciclo anual. El mismo patrón se repite en el resto de parámetros. La asociación aparente procede por tanto de la variación entre embalses y de la estacionalidad compartida por ambas series, no de una relación temporal entre ellas.

Frente a la variación futura del llenado, la mediana de los coeficientes se sitúa entre −0,053 y +0,038 en los tres horizontes, valores indistinguibles de cero. El examen individual identifica coeficientes de mayor magnitud en algunos embalses, con un máximo de 0,222 correspondiente al oxígeno disuelto en Leboreiro Mao y Edrada Mao a treinta días. Su interpretación exige cautela por dos motivos: ambos embalses comparten la misma estación de calidad asignada, por lo que no constituyen observaciones independientes, y el signo de la relación no es consistente entre embalses, ya que el mismo parámetro presenta coeficiente
positivo en Bárcena.

El análisis acota lo que cabe esperar del diseño experimental. En modelos lineales en sus regresores, como SARIMAX, no es previsible que el bloque de calidad aporte capacidad predictiva por encima de la que ya proporcionan la propia serie de llenado y las variables meteorológicas e hidrológicas. La posibilidad de relaciones no lineales o condicionadas a otras variables permanece abierta para los modelos de aprendizaje automático, cuya capacidad para capturarlas motiva precisamente su inclusión en el trabajo.

Conviene subrayar que este resultado no invalida el planteamiento, sino que lo sitúa: la aportación del trabajo reside en someter la hipótesis a una comprobación rigurosa y documentar el resultado, sea cual sea su signo.